In [1]:
import os
import torch
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [2]:
class CIFAR10CustomDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None, is_test=False):
        """
        Args:
            csv_file (string): Path to the csv file with annotations.
            img_dir (string): Directory with all the images.
            transform (callable, optional): Optional transform to be applied on a sample.
            is_test (bool): Set to True if loading the unlabeled test dataset.
        """
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        # Fetch the filename from the CSV
        img_name = self.df.iloc[index]['filename']
        img_path = os.path.join(self.img_dir, img_name)
        
        # Load the image and ensure it stays RGB (3 channels)
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
            
        if self.is_test:
            # For the test/evaluation dataset, return the image and its unique ID
            img_id = self.df.iloc[index]['id']
            return image, img_id
        else:
            # For the training dataset, return the image and its integer label (0-9)
            label = int(self.df.iloc[index]['label'])
            return image, torch.tensor(label, dtype=torch.long)


In [ ]:
# --- 1. Transforms & Augmentations ---
train_transforms = transforms.Compose([
    transforms.RandomCrop(32, padding=4),       # Standard CIFAR padding-crop augmentation
    transforms.RandomHorizontalFlip(),          # Flips the image horizontally 50% of the time
    transforms.ToTensor(),                      # Scales pixels to [0.0, 1.0] and converts to tensor
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], 
                         std=[0.2023, 0.1994, 0.2010]) # Standard CIFAR-10 normalization stats
])

test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2023, 0.1994, 0.2010])
])

# --- 2. Dataset Instantiation ---
train_dataset = CIFAR10CustomDataset(csv_file='./data/train_labels.csv', img_dir='./data/train_images', transform=train_transforms, is_test=False)
test_dataset = CIFAR10CustomDataset(csv_file='./data/test.csv', img_dir='./data/test_images', transform=test_transforms, is_test=True)

# --- 3. Wrap in DataLoaders ---
train_loader = DataLoader(
    dataset=train_dataset, 
    batch_size=128, 
    shuffle=True,            # Shuffle to break structural ordering patterns during training
    num_workers=4, 
    pin_memory=True          # Enables faster data transfer when calling images.to(device)
)

test_loader = DataLoader(
    dataset=test_dataset, 
    batch_size=128, 
    shuffle=False, 
    num_workers=4, 
    pin_memory=True
)